## 1. Instalação de Dependências

Nesta seção, instalamos todas as bibliotecas Python necessárias para o desenvolvimento do nosso sistema de detecção de sirenes. As principais bibliotecas incluem:

- `soundata`: Para fácil download e gerenciamento do dataset UrbanSound8K.
- `librosa`: Essencial para o processamento e extração de características de áudio.
- `numpy` e `scikit-learn`: Para manipulação de dados e ferramentas de machine learning.
- `tensorflow`: Para construir e treinar o modelo de classificação.
- `tf2onnx`: Para converter o modelo Keras em formato ONNX.

`pip install -q` garante uma instalação silenciosa, sem muita saída no console.

In [ ]:
!pip install soundata librosa numpy scikit-learn tensorflow tf2onnx -q

## 2. Download e Validação do Dataset UrbanSound8K

Aqui, utilizamos a biblioteca `soundata` para gerenciar o dataset UrbanSound8K. Este dataset é fundamental para treinar e testar nosso modelo de detecção de sirenes. Os passos são:

1.  **Inicialização**: `soundata.initialize('urbansound8k')` configura o dataset.
2.  **Download**: `dataset.download()` baixa os arquivos de áudio e metadados, se ainda não estiverem presentes.
3.  **Validação**: `dataset.validate()` verifica a integridade dos arquivos baixados.
4.  **Carregamento dos Clips**: `dataset.load_clips()` carrega os metadados de cada clipe de áudio, permitindo acesso a informações como `class_label`, `fold`, e `audio_path`.

Exibimos o número total de clips e um exemplo para entender a estrutura dos dados.

In [ ]:
import soundata

dataset = soundata.initialize('urbansound8k')
dataset.download()
dataset.validate()

clips = dataset.load_clips()
print(f"Total de clips: {len(clips)}")

example_clip = dataset.choice_clip()
print(example_clip)

## 3. Definição de Parâmetros Fixos do Pipeline

Esta seção define um conjunto de parâmetros globais que são cruciais para todo o pipeline, desde a extração de características até o treinamento e a exportação do modelo. É fundamental que esses valores sejam **IDÊNTICOS** aos que serão utilizados no firmware do ESP32 (Task 2) para garantir compatibilidade e consistência na detecção.

Os parâmetros incluem:

-   `SAMPLE_RATE`: Taxa de amostragem do áudio (em Hz).
-   `WINDOW_SIZE`: Número de amostras por janela de áudio (tamanho do frame).
-   `HOP_SIZE`: Número de amostras de "salto" entre janelas (overlap).
-   `N_MFCC`: Número de coeficientes MFCC a serem extraídos.
-   `N_FFT`: Tamanho da Transformada Rápida de Fourier (geralmente igual ao `WINDOW_SIZE`).
-   `POSITIVE_CLASS`: A classe que queremos detectar (neste caso, "siren").
-   `FOLDS_TRAIN`, `FOLD_VAL`, `FOLD_TEST`: Divisão dos folds do dataset UrbanSound8K para treinamento, validação e teste na validação cruzada.
-   `RANDOM_SEED`: Semente para reprodutibilidade dos resultados.

In [ ]:
SAMPLE_RATE = 16000
WINDOW_SIZE = 1024
HOP_SIZE = 512
N_MFCC = 10
N_FFT = 1024

POSITIVE_CLASS = "siren"
FOLDS_TRAIN = list(range(1, 9))
FOLD_VAL = 9
FOLD_TEST = 10

RANDOM_SEED = 42

## 4. Separação e Balanceamento dos Clips por Fold

Nesta etapa, organizamos os clips de áudio do dataset UrbanSound8K de acordo com seus respectivos *folds* (subconjuntos) e classes (positiva ou negativa em relação à sirene). Isso é essencial para realizar uma validação cruzada adequada e para treinar um modelo balanceado.

1.  **Organização Inicial**: Os clips são iterados, e cada um é categorizado em `clips_by_fold` como "positive" (se for uma sirene) ou "negative" (se não for).
2.  **Balanceamento**: Para evitar que o modelo seja viesado para a classe majoritária (não-sirene), realizamos um balanceamento. Para cada fold, amostramos um número de clips "negative" igual ao número de clips "positive". Isso garante que o modelo veja uma quantidade equilibrada de exemplos de cada classe durante o treinamento.

O output mostra a quantidade de clips positivos e negativos selecionados para cada fold após o balanceamento.

In [ ]:
import random
random.seed(RANDOM_SEED)

clips_by_fold = {i: {"positive": [], "negative": []} for i in range(1, 11)}

for clip_id, clip in clips.items():
    fold = clip.fold
    class_label = clip.class_label
    if class_label == POSITIVE_CLASS:
        clips_by_fold[fold]["positive"].append(clip)
    else:
        clips_by_fold[fold]["negative"].append(clip)

balanced_clips_by_fold = {}
for fold, groups in clips_by_fold.items():
    n_pos = len(groups["positive"])
    n_neg_available = len(groups["negative"])
    n_neg_sample = min(n_pos, n_neg_available)  # balanceia
    sampled_negatives = random.sample(groups["negative"], n_neg_sample)
    balanced_clips_by_fold[fold] = {
        "positive": groups["positive"],
        "negative": sampled_negatives
    }
    print(f"Fold {fold}: {n_pos} positivos, {n_neg_sample} negativos (de {n_neg_available})")

## 5. Funções de Extração de Features por Janela

Esta seção define as funções responsáveis por carregar e extrair características de áudio de cada clipe. A extração de features é um passo crítico para transformar os dados de áudio brutos em uma representação numérica que o modelo de machine learning possa entender.

-   `extract_features_from_audio`: Esta função recebe o áudio, a taxa de amostragem e os parâmetros de janela, e calcula as seguintes características para cada janela de áudio:
    -   **RMS (Root Mean Square)**: Medida da energia do sinal de áudio.
    -   **Centróide Espectral**: Indica onde o "centro" de massa do espectro de potência está localizado, ou seja, onde a maior parte da energia espectral está concentrada.
    -   **MFCCs (Mel-frequency Cepstral Coefficients)**: Coeficientes que representam o envelope espectral de um sinal de áudio, importantes para reconhecimento de fala e música. `N_MELS` define o número de bandas mel usadas na transformação.

-   `load_and_process_clip`: Esta função de conveniência carrega um arquivo de áudio usando `librosa`, realiza o resample se necessário para a `SAMPLE_RATE` definida, e então chama `extract_features_from_audio` para obter as características.

In [ ]:
import numpy as np
import librosa

N_MELS = 26

def extract_features_from_audio(audio, sr, window_size, hop_size, n_mfcc, n_fft):
    if len(audio) < window_size:
        audio = np.pad(audio, (0, window_size - len(audio)))

    features_list = []

    for start in range(0, len(audio) - window_size + 1, hop_size):
        window = audio[start:start + window_size]

        rms = np.sqrt(np.mean(window**2))

        centroid = librosa.feature.spectral_centroid(
            y=window, sr=sr, n_fft=n_fft, hop_length=window_size
        )[0, 0]

        mfccs = librosa.feature.mfcc(
            y=window, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=window_size,
            n_mels=N_MELS
        )[:, 0]

        feature_vector = np.concatenate(([rms, centroid], mfccs))
        features_list.append(feature_vector)

    return np.array(features_list)

def load_and_process_clip(clip, sample_rate, window_size, hop_size, n_mfcc, n_fft):
    """
    Carrega um clip de áudio, resample para sample_rate, e extrai features.
    """
    audio, orig_sr = librosa.load(clip.audio_path, sr=None, mono=True)
    if orig_sr != sample_rate:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sample_rate)

    return extract_features_from_audio(
        audio, sample_rate, window_size, hop_size, n_mfcc, n_fft
    )

## 6. Processamento dos Clips e Montagem dos Datasets por Fold

Nesta etapa, as funções de extração de características definidas anteriormente são aplicadas a todos os clips de áudio balanceados. O objetivo é construir os conjuntos de dados (X para features, y para labels) para cada fold, que serão usados na validação cruzada.

-   `build_feature_dataset`: Esta função itera sobre os clips positivos e negativos de uma lista de folds, carrega e processa cada um deles para extrair as características. Para cada janela de áudio, um vetor de características (`feats`) é gerado, e um label (`1` para sirene, `0` para não-sirene) é associado.
-   Os vetores de características e os labels são concatenados para formar os arrays `X` e `y` para cada fold. Um mecanismo de tratamento de exceções (`try-except`) é incluído para lidar com possíveis erros durante o processamento de clips específicos.

O output da célula `qbHpAyuv4WXD` mostrará as dimensões dos conjuntos `X` e `y` para cada fold, indicando quantos vetores de características foram extraídos e seus labels correspondentes.

In [ ]:
from tqdm import tqdm

def build_feature_dataset(fold_list, balanced_clips_by_fold):
    """
    Para uma lista de folds, extrai features de todos os clips (positivos e negativos)
    e retorna X (features) e y (labels: 1=sirene, 0=não-sirene).
    """
    X_list = []
    y_list = []

    for fold in fold_list:
        for clip in tqdm(balanced_clips_by_fold[fold]["positive"], desc=f"Fold {fold} - positivos"):
            try:
                feats = load_and_process_clip(clip, SAMPLE_RATE, WINDOW_SIZE, HOP_SIZE, N_MFCC, N_FFT)
                X_list.append(feats)
                y_list.append(np.ones(len(feats)))
            except Exception as e:
                print(f"Erro no clip {clip.clip_id}: {e}")

        for clip in tqdm(balanced_clips_by_fold[fold]["negative"], desc=f"Fold {fold} - negativos"):
            try:
                feats = load_and_process_clip(clip, SAMPLE_RATE, WINDOW_SIZE, HOP_SIZE, N_MFCC, N_FFT)
                X_list.append(feats)
                y_list.append(np.zeros(len(feats)))
            except Exception as e:
                print(f"Erro no clip {clip.clip_id}: {e}")

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y

In [ ]:
fold_features_cache = {}

for fold in range(1, 11):
    X_fold, y_fold = build_feature_dataset([fold], balanced_clips_by_fold)
    fold_features_cache[fold] = (X_fold, y_fold)
    print(f"Fold {fold}: X={X_fold.shape}, y={y_fold.shape}")

## 7. Construção do Modelo Keras (Rede Neural)

Esta seção define a arquitetura da rede neural Keras que será utilizada para classificar as características de áudio. O modelo é uma rede neural densa (Fully Connected) simples, projetada para ser eficiente e compacta, adequada para implantação em dispositivos embarcados como o ESP32.

-   `build_model(n_features, dropout_rate)`:
    -   Recebe `n_features` (o número de características de entrada) e `dropout_rate` (para regularização).
    -   **Camada de Entrada**: Define o formato da entrada com base no número de features.
    -   **Camadas Ocultas**: Duas camadas densas com 16 e 8 neurônios, respectivamente, utilizando a função de ativação ReLU (`activation='relu'`).
    -   **Camadas de Dropout**: Inseridas após cada camada densa para prevenir overfitting, desativando aleatoriamente uma fração dos neurônios durante o treinamento.
    -   **Camada de Saída**: Uma única camada densa com um neurônio e função de ativação sigmoide (`activation='sigmoid'`), que produz uma probabilidade de a entrada pertencer à classe positiva (sirene).
    -   **Compilação do Modelo**: O modelo é compilado com o otimizador Adam, `binary_crossentropy` como função de perda (adequada para classificação binária) e métricas como acurácia, precisão e recall para monitoramento.

In [ ]:
import tensorflow as tf
from tensorflow import keras

def build_model(n_features, dropout_rate=0.3):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(8, activation='relu'),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
    )
    return model

## 8. Treinamento e Avaliação do Modelo via Validação Cruzada (10 Folds)

Nesta seção, o modelo Keras é treinado e avaliado usando um esquema de validação cruzada com 10 folds. Este método ajuda a obter uma estimativa mais robusta da performance do modelo, reduzindo o viés de uma única divisão de treino/teste.

Para cada um dos 10 folds:

1.  **Divisão dos Dados**: Os dados são divididos em:
    -   **9 Folds para Treino**: Usados para treinar o modelo.
    -   **1 Fold para Validação**: Usado para monitorar a performance durante o treinamento e aplicar `EarlyStopping`.
    -   **1 Fold para Teste**: Usado para a avaliação final do modelo naquele ciclo de validação cruzada.

2.  **Normalização**: Os dados de treino, validação e teste são normalizados (padronizados para média zero e desvio padrão um) usando as estatísticas (média e desvio padrão) calculadas **APENAS** no conjunto de treino. Isso simula um cenário real onde as estatísticas dos dados de teste são desconhecidas.

3.  **Construção e Treinamento do Modelo**: Um novo modelo Keras é construído (`build_model`) e treinado usando os dados normalizados de treino e validação. Um callback `EarlyStopping` é configurado para interromper o treinamento se a `val_loss` não melhorar por 10 épocas, restaurando os melhores pesos.

4.  **Avaliação**: Após o treinamento, o modelo é avaliado no conjunto de teste para calcular as métricas de desempenho:
    -   **Acurácia**: Proporção de predições corretas.
    -   **Precisão**: Proporção de positivos verdadeiros entre todos os positivos preditos.
    -   **Recall**: Proporção de positivos verdadeiros entre todos os positivos reais.
    -   **ROC-AUC**: Área sob a Curva Característica de Operação do Receptor, uma métrica robusta para classificação binária.

Os resultados de cada fold de teste são armazenados em `cv_results` e impressos no final de cada iteração.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

cv_results = []
all_folds = list(range(1, 11))

for test_fold in all_folds:
    val_fold = all_folds[(all_folds.index(test_fold) + 1) % 10]
    train_folds = [f for f in all_folds if f not in (test_fold, val_fold)]

    X_train_cv = np.concatenate([fold_features_cache[f][0] for f in train_folds], axis=0)
    y_train_cv = np.concatenate([fold_features_cache[f][1] for f in train_folds], axis=0)
    X_val_cv, y_val_cv = fold_features_cache[val_fold]
    X_test_cv, y_test_cv = fold_features_cache[test_fold]

    mean_cv = X_train_cv.mean(axis=0)
    std_cv = X_train_cv.std(axis=0)
    std_cv[std_cv == 0] = 1e-8

    X_train_cv_norm = (X_train_cv - mean_cv) / std_cv
    X_val_cv_norm = (X_val_cv - mean_cv) / std_cv
    X_test_cv_norm = (X_test_cv - mean_cv) / std_cv

    model_cv = build_model(X_train_cv_norm.shape[1])

    early_stop_cv = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    )

    model_cv.fit(
        X_train_cv_norm, y_train_cv,
        validation_data=(X_val_cv_norm, y_val_cv),
        epochs=100, batch_size=32,
        callbacks=[early_stop_cv],
        verbose=0
    )

    y_pred_proba_cv = model_cv.predict(X_test_cv_norm, verbose=0).flatten()
    y_pred_cv = (y_pred_proba_cv >= 0.5).astype(int)

    result = {
        "test_fold": test_fold,
        "accuracy": accuracy_score(y_test_cv, y_pred_cv),
        "precision": precision_score(y_test_cv, y_pred_cv),
        "recall": recall_score(y_test_cv, y_pred_cv),
        "roc_auc": roc_auc_score(y_test_cv, y_pred_proba_cv),
    }
    cv_results.append(result)
    print(f"Fold {test_fold} (teste) | val={val_fold} | "
          f"acc={result['accuracy']:.4f} prec={result['precision']:.4f} "
          f"rec={result['recall']:.4f} auc={result['roc_auc']:.4f}")

## 9. Agregação e Visualização dos Resultados da Validação Cruzada

Após o processo de validação cruzada de 10 folds, esta seção agrega e visualiza os resultados para fornecer uma visão geral da performance do modelo.

1.  **DataFrame de Resultados**: Os resultados coletados em `cv_results` (acurácia, precisão, recall, ROC-AUC para cada fold de teste) são convertidos em um DataFrame do Pandas para facilitar a análise.
2.  **Resumo Estatístico**: É calculada e impressa a média e o desvio padrão de cada métrica em todos os 10 folds. Isso oferece uma medida da performance geral do modelo e da sua variabilidade entre as diferentes partições dos dados.
3.  **Visualização**: Um gráfico de barras é gerado para exibir as métricas de desempenho para cada fold de teste individualmente. Isso permite identificar se o modelo performou de maneira consistente em todos os folds ou se houve alguma flutuação significativa (por exemplo, um fold onde o modelo teve um desempenho particularmente baixo, como o fold 10 neste caso, que pode indicar um subconjunto de dados mais desafiador).

In [ ]:
import pandas as pd

df_cv = pd.DataFrame(cv_results)
print(df_cv)

print("\n=== Resumo (média ± desvio padrão entre os 10 folds) ===")
for metric in ["accuracy", "precision", "recall", "roc_auc"]:
    print(f"{metric}: {df_cv[metric].mean():.4f} ± {df_cv[metric].std():.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
df_cv.set_index("test_fold")[["accuracy", "precision", "recall", "roc_auc"]].plot(kind="bar", ax=ax)
ax.set_title("Métricas por fold de teste (Cross-Validation)")
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 10. Combinação dos Folds e Separação de Holdout para Early Stopping

Com a avaliação por validação cruzada concluída, agora preparamos os dados para o treinamento do **modelo final**. O objetivo é treinar o modelo com a maior quantidade de dados possível para maximizar seu desempenho, mas ainda ter um pequeno conjunto para monitorar o treinamento e evitar overfitting.

1.  **Combinação de Todos os Folds**: Todos os 10 folds de características (`X_all`) e labels (`y_all`) são concatenados. Este será o conjunto completo de dados que o modelo final verá durante o treinamento.
2.  **Separação de um Holdout Estratificado**: Um pequeno subconjunto (5%) dos dados combinados é separado como um conjunto de *holdout*. Este `X_holdout`, `y_holdout` será usado exclusivamente como conjunto de validação para o `EarlyStopping` durante o treinamento do modelo final. A separação é `estratificada` (`stratify=y_all`) para garantir que a proporção de classes (sirene vs. não-sirene) seja mantida no conjunto de treino final e no holdout, evitando desequilíbrios.

Este `holdout` é diferente do conjunto de validação usado na validação cruzada; ele serve apenas para guiar o `EarlyStopping` e não para avaliação final da performance geral, que já foi obtida na validação cruzada.

In [ ]:
from sklearn.model_selection import train_test_split

X_all = np.concatenate([fold_features_cache[f][0] for f in range(1, 11)], axis=0)
y_all = np.concatenate([fold_features_cache[f][1] for f in range(1, 11)], axis=0)

print(f"Total combinado: X={X_all.shape}, y={y_all.shape}")

X_train_final, X_holdout, y_train_final, y_holdout = train_test_split(
    X_all, y_all, test_size=0.05, stratify=y_all, random_state=RANDOM_SEED
)

print(f"Treino final: {X_train_final.shape} | Holdout: {X_holdout.shape}")

## 11. Normalização Definitiva dos Dados

Esta é uma etapa crítica para o treinamento do modelo final e, mais importante, para a sua implantação no ESP32. Calculamos as estatísticas de normalização (média e desvio padrão) com base no conjunto de treino final (`X_train_final`).

-   **`feature_mean`**: A média de cada característica em `X_train_final`.
-   **`feature_std`**: O desvio padrão de cada característica em `X_train_final`.
-   A linha `std_cv[std_cv == 0] = 1e-8` é um tratamento para evitar divisão por zero caso alguma característica tenha desvio padrão nulo (todos os valores são iguais).

Esses valores (`feature_mean` e `feature_std`) são os **DEFINITIVOS** que deverão ser usados para normalizar os dados de entrada no firmware do ESP32. O ESP32 receberá dados brutos do microfone, extrairá as mesmas características e as normalizará com *exatamente* esses `feature_mean` e `feature_std` antes de passá-los para o modelo TFLite. O output mostra esses valores para posterior inclusão no código C.

In [ ]:
feature_mean = X_train_final.mean(axis=0)
feature_std = X_train_final.std(axis=0)
feature_std[feature_std == 0] = 1e-8

X_train_norm = (X_train_final - feature_mean) / feature_std
X_holdout_norm = (X_holdout - feature_mean) / feature_std

print("feature_mean =", feature_mean.tolist())
print("feature_std =", feature_std.tolist())

## 12. Treinamento do Modelo Final

Nesta seção, o modelo Keras é treinado uma última vez, utilizando o conjunto de dados mais robusto (todos os folds combinados, exceto o pequeno holdout para early stopping) e os parâmetros de normalização definitivos.

1.  **Inicialização do Modelo**: Um novo modelo Keras é construído usando a função `build_model` definida anteriormente, com o número de características correto.
2.  **Callback `EarlyStopping`**: Configurado para monitorar a `val_loss` (perda no conjunto de holdout) e parar o treinamento se não houver melhoria após 10 épocas, restaurando os pesos do modelo que teve a melhor performance.
3.  **Treinamento**: O modelo é treinado usando `X_train_norm` e `y_train_final` (os dados de treino normalizados) e validado com `X_holdout_norm` e `y_holdout` (o conjunto de holdout normalizado). O treinamento é verbose para que possamos acompanhar o progresso.

Este modelo treinado (`model`) é o que será subsequentemente exportado para o formato ONNX e quantizado para TFLite Micro para implantação no ESP32.

In [ ]:
n_features = X_train_norm.shape[1]

model = build_model(n_features)

early_stop_final = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history_final = model.fit(
    X_train_norm, y_train_final,
    validation_data=(X_holdout_norm, y_holdout),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_final],
    verbose=1
)

## 13. Avaliação de Sanidade no Holdout do Modelo Final

Após o treinamento do modelo final, realizamos uma avaliação de "sanidade" no conjunto de holdout que foi separado anteriormente. Este passo verifica se o modelo final se comporta conforme o esperado em dados não vistos durante o treinamento principal.

1.  **Predições**: O modelo final (`model`) faz predições de probabilidade (`y_pred_proba_final`) no conjunto de holdout normalizado (`X_holdout_norm`). Essas probabilidades são então binarizadas (`y_pred_final`) usando um limiar de 0.5 para obter as classes preditas.
2.  **Relatório de Classificação**: `classification_report` do scikit-learn é usado para gerar um relatório detalhado das métricas de precisão, recall e f1-score para cada classe (não-sirene e sirene).
3.  **Matriz de Confusão**: `confusion_matrix` é impressa para visualizar os verdadeiros positivos, verdadeiros negativos, falsos positivos e falsos negativos, oferecendo uma compreensão clara de onde o modelo está acertando e errando.
4.  **ROC-AUC**: O `roc_auc_score` é calculado para dar uma métrica agregada da performance do modelo, especialmente útil para problemas de classificação binária.

Este relatório é um último check para garantir que o modelo treinado está performando de forma aceitável antes de prosseguir com a exportação e quantização para o ESP32.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred_proba_final = model.predict(X_holdout_norm).flatten()
y_pred_final = (y_pred_proba_final >= 0.5).astype(int)

print("=== Modelo Final — Avaliação no Holdout ===")
print(classification_report(y_holdout, y_pred_final, target_names=["não-sirene", "sirene"]))
print("Matriz de confusão:")
print(confusion_matrix(y_holdout, y_pred_final))
print(f"ROC-AUC: {roc_auc_score(y_holdout, y_pred_proba_final):.4f}")

## 14. Exportação do Modelo para ONNX

Esta seção foca na exportação do modelo Keras treinado para o formato ONNX (Open Neural Network Exchange). O ONNX é um formato aberto e interoperável que permite que modelos de machine learning sejam movidos entre diferentes frameworks (como TensorFlow, PyTorch, Caffe2) e plataformas de hardware.

-   **`tf2onnx.convert.from_function`**: É utilizada para converter o modelo Keras. Para modelos Keras 3, é recomendável envolver o modelo em uma `tf.function` com uma `input_signature` explícita. Isso define o formato de entrada esperado pelo modelo ONNX, o que é crucial para ferramentas downstream.
-   **`output_path="modelo_sirene.onnx"`**: O modelo convertido é salvo em um arquivo chamado `modelo_sirene.onnx`.

Embora o ONNX não seja diretamente usado no ESP32 (que usará TFLite Micro), ele serve como um formato intermediário robusto e universal, que pode ser útil para outras ferramentas ou otimizações futuras, ou mesmo para uma etapa de conversão posterior para TFLite caso a conversão direta de Keras para TFLite apresente problemas.

In [ ]:
import tf2onnx
import tensorflow as tf

@tf.function(input_signature=[tf.TensorSpec((None, n_features), tf.float32, name="input")])
def model_fn(x):
    return {"output": model(x)}

model_proto, _ = tf2onnx.convert.from_function(
    model_fn,
    input_signature=[tf.TensorSpec((None, n_features), tf.float32, name="input")],
    output_path="modelo_sirene.onnx"
)

print("Modelo ONNX salvo em modelo_sirene.onnx")

## 15. Quantização Pós-Treinamento (Post-Training Quantization) para TFLite Micro

Para otimizar o modelo para implantação em microcontroladores como o ESP32, que possuem recursos de memória e processamento limitados, realizamos a quantização pós-treinamento. Este processo reduz a precisão dos pesos e ativações do modelo (geralmente de float32 para int8), diminuindo o tamanho do modelo e acelerando a inferência, com um impacto mínimo na acurácia.

1.  **`representative_dataset_gen`**: Uma função geradora é definida para fornecer um pequeno conjunto de dados representativo do treino (`X_train_norm`). Este conjunto é usado pelo conversor TFLite para calibrar a quantização, determinando os ranges de valores para mapear para int8.
2.  **`tf.lite.TFLiteConverter.from_keras_model(model)`**: Inicializa o conversor com o modelo Keras treinado.
3.  **`converter.optimizations = [tf.lite.Optimize.DEFAULT]`**: Habilita as otimizações padrão do TFLite, que incluem a quantização.
4.  **`converter.representative_dataset = representative_dataset_gen`**: Atribui o conjunto de dados representativo ao conversor.
5.  **`converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]`**: Força a quantização full-integer (int8), que é essencial para o TFLite Micro.
6.  **`converter.inference_input_type` e `converter.inference_output_type`**: Especificam que as entradas e saídas do modelo quantizado também serão int8.
7.  **`tflite_quant_model = converter.convert()`**: Realiza a conversão e quantização do modelo.
8.  **Salvamento**: O modelo TFLite quantizado é salvo em `modelo_sirene_quant.tflite` e seu tamanho em bytes é impresso, demonstrando a redução de tamanho.

In [ ]:
def representative_dataset_gen():
    n_samples = min(300, len(X_train_norm))
    indices = np.random.choice(len(X_train_norm), n_samples, replace=False)
    for i in indices:
        sample = X_train_norm[i:i+1].astype(np.float32)
        yield [sample]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

with open("modelo_sirene_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

print(f"Modelo quantizado salvo: {len(tflite_quant_model)} bytes")

## 16. Validação da Acurácia do Modelo Quantizado

Após a quantização, é crucial verificar se o modelo quantizado (int8) mantém um nível de acurácia aceitável em comparação com o modelo float32 original. Esta seção carrega o modelo quantizado e testa sua performance no mesmo conjunto de holdout usado anteriormente.

1.  **Carregamento do Intérprete TFLite**: O modelo TFLite quantizado é carregado usando `tf.lite.Interpreter`. Os detalhes de entrada e saída (incluindo escala e ponto zero da quantização) são extraídos.
2.  **Função `predict_quantized`**: Esta função simula como o ESP32 fará as predições. Ela converte a entrada float32 para int8 usando os parâmetros de quantização, executa a inferência no intérprete TFLite e, em seguida, converte a saída int8 de volta para float32.
3.  **Predições no Holdout**: A função `predict_quantized` é aplicada a todo o `X_holdout_norm` para obter as predições do modelo quantizado.
4.  **Relatório de Classificação**: Um `classification_report` é gerado para o modelo quantizado, permitindo uma análise detalhada de sua performance.
5.  **Comparação de Acurácia**: A acurácia do modelo quantizado é comparada diretamente com a acurácia do modelo float32 original (calculada novamente no mesmo holdout para uma comparação justa). Isso nos permite quantificar qualquer perda de precisão devido à quantização.

Este passo garante que a otimização de tamanho e velocidade do modelo não comprometeu sua capacidade de classificar sirenes de forma eficaz.

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale, input_zero_point = input_details['quantization']
output_scale, output_zero_point = output_details['quantization']

print("Input details:", input_details)
print("Output details:", output_details)

def predict_quantized(x_float):
    x_int8 = (x_float / input_scale + input_zero_point).astype(np.int8)
    interpreter.set_tensor(input_details['index'], x_int8.reshape(1, -1))
    interpreter.invoke()
    out_int8 = interpreter.get_tensor(output_details['index'])
    out_float = (out_int8.astype(np.float32) - output_zero_point) * output_scale
    return out_float[0, 0]

y_pred_quant = np.array([predict_quantized(x) for x in X_holdout_norm.astype(np.float32)])
y_pred_quant_binary = (y_pred_quant >= 0.5).astype(int)

print("\n=== Modelo Quantizado (int8) — avaliado no holdout correto ===")
print(classification_report(y_holdout, y_pred_quant_binary, target_names=["não-sirene", "sirene"]))

y_pred_float_holdout = (model.predict(X_holdout_norm).flatten() >= 0.5).astype(int)
acc_float_holdout = (y_pred_float_holdout == y_holdout).mean()

print("\n=== Comparação (mesmo holdout) ===")
print(f"Accuracy float:      {acc_float_holdout:.4f}")
print(f"Accuracy quantizado: {(y_pred_quant_binary == y_holdout).mean():.4f}")

## 17. Geração do Array C para o Firmware (Modelo TFLite Micro)

Esta seção é vital para a implantação do modelo em um microcontrolador como o ESP32. O modelo TFLite quantizado é convertido em um array de bytes em formato C, que pode ser diretamente incluído no código-fonte do firmware.

-   **`convert_to_c_array`**: Esta função lê o arquivo `.tflite` binário, itera sobre seus bytes e os escreve em um arquivo `.h` (header C) no formato de um array `const unsigned char []`.
-   **`alignas(8)`**: Garante que o array de bytes seja alinhado na memória em um limite de 8 bytes, o que pode ser importante para a performance em algumas arquiteturas de microcontroladores.
-   **Nome da Variável**: O array é nomeado `modelo_sirene_tflite`, e seu tamanho (`modelo_sirene_tflite_len`) também é exportado.
-   **Guarda de Inclusão (`#ifndef`, `#define`, `#endif`)**: Padrão para arquivos de cabeçalho C para evitar múltiplas inclusões.

O arquivo `modelo_sirene.h` gerado conterá a representação binária do modelo TFLite Micro, pronta para ser compilada junto com o firmware do ESP32, eliminando a necessidade de carregar o modelo de um sistema de arquivos ou armazenamento externo.

In [ ]:
def convert_to_c_array(tflite_file, output_header):
    with open(tflite_file, 'rb') as f:
        data = f.read()

    var_name = "modelo_sirene_tflite"
    with open(output_header, 'w') as f:
        f.write(f"// Modelo TFLite Micro gerado automaticamente\n")
        f.write(f"// Tamanho: {len(data)} bytes\n\n")
        f.write(f"#ifndef MODELO_SIRENE_H\n#define MODELO_SIRENE_H\n\n")
        f.write(f"alignas(8) const unsigned char {var_name}[] = {{\n")

        for i, byte in enumerate(data):
            f.write(f"0x{byte:02x}, ")
            if (i + 1) % 12 == 0:
                f.write("\n")

        f.write(f"\n}};\n\n")
        f.write(f"const unsigned int {var_name}_len = {len(data)};\n\n")
        f.write(f"#endif\n")

    print(f"Header gerado: {output_header} ({len(data)} bytes)")

convert_to_c_array("modelo_sirene_quant.tflite", "modelo_sirene.h")

## 18. Salvar Parâmetros de Normalização e Quantização em Formato C

Complementando a exportação do modelo, esta seção gera um segundo arquivo de cabeçalho C contendo todos os parâmetros numéricos necessários para a etapa de pré-processamento e pós-processamento no firmware do ESP32.

-   **`generate_normalization_header`**: Esta função recebe:
    -   `feature_mean` e `feature_std`: As médias e desvios padrão definitivos das características, calculados no treino final (ver Seção 11).
    -   `input_scale`, `input_zero_point`, `output_scale`, `output_zero_point`: Os parâmetros de quantização para entrada e saída do modelo, obtidos do modelo TFLite quantizado (ver Seção 15).
-   **Arquivo `normalizacao_params.h`**: Um arquivo de cabeçalho C é gerado com `define`s e `const float []` para esses parâmetros.
-   **Formato Float (`f`)**: Os valores de ponto flutuante são exportados com o sufixo `f` (ex: `0.12345f`) para garantir que sejam tratados como `float` (32-bit) em C, correspondendo à precisão dos cálculos no microcontrolador.

Com este arquivo, o firmware do ESP32 terá todas as informações necessárias para:
1.  Normalizar as características extraídas do áudio, usando os mesmos `mean` e `std` do treinamento.
2.  Quantizar a entrada para o modelo TFLite Micro (usando `INPUT_SCALE` e `INPUT_ZERO_POINT`).
3.  Desquantizar a saída do modelo (usando `OUTPUT_SCALE` e `OUTPUT_ZERO_POINT`) para obter a probabilidade final.

In [ ]:
def generate_normalization_header(feature_mean, feature_std, input_scale, input_zero_point,
                                     output_scale, output_zero_point, filename="normalizacao_params.h"):
    with open(filename, 'w') as f:
        f.write("#ifndef NORMALIZACAO_PARAMS_H\n#define NORMALIZACAO_PARAMS_H\n\n")
        f.write(f"#define N_FEATURES {len(feature_mean)}\n\n")

        f.write("const float feature_mean[N_FEATURES] = {")
        f.write(", ".join(f"{v:.8f}f" for v in feature_mean))
        f.write("};\n\n")

        f.write("const float feature_std[N_FEATURES] = {")
        f.write(", ".join(f"{v:.8f}f" for v in feature_std))
        f.write("};\n\n")

        f.write(f"const float INPUT_SCALE = {input_scale:.10f}f;\n")
        f.write(f"const int INPUT_ZERO_POINT = {input_zero_point};\n")
        f.write(f"const float OUTPUT_SCALE = {output_scale:.10f}f;\n")
        f.write(f"const int OUTPUT_ZERO_POINT = {output_zero_point};\n\n")

        f.write("#endif\n")

    print(f"Header gerado: {filename}")

generate_normalization_header(
    feature_mean, feature_std,
    input_scale, input_zero_point,
    output_scale, output_zero_point
)

## 19. Exportação de Mel Filterbank e Matriz DCT para C

Para garantir que a extração de características (MFCCs) no firmware do ESP32 seja idêntica à que foi feita durante o treinamento, é necessário exportar as matrizes `Mel Filterbank` e `DCT` (Discrete Cosine Transform) para o formato C.

-   **`mel_filterbank`**: Calculado usando `librosa.filters.mel` com os mesmos `SAMPLE_RATE`, `N_FFT` e `N_MELS` definidos nos parâmetros fixos. Esta matriz é usada para transformar o espectro de potência em um espectro em escala mel.
-   **`dct_matrix`**: Gerada a partir da função `dct` da `scipy.fftpack`, representando a transformação DCT tipo II. Esta matriz é aplicada ao log-mel-espectro para obter os coeficientes MFCC.

-   **`export_matrix_c`**: Uma função auxiliar é definida para escrever essas matrizes em arquivos de cabeçalho C (`mel_filterbank.h` e `dct_matrix.h`). As matrizes são exportadas como arrays `const float[][]`, juntamente com as definições de suas dimensões (`ROWS` e `COLS`).

Com esses arquivos, o firmware terá as tabelas de lookup exatas para replicar o processo de extração de características, garantindo que as entradas para o modelo TFLite Micro sejam consistentes com as usadas no treinamento.

In [ ]:
import numpy as np
from scipy.fftpack import dct

mel_filterbank = librosa.filters.mel(sr=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS)

dct_matrix = dct(np.eye(N_MELS), type=2, norm='ortho', axis=0)[:N_MFCC, :]

def export_matrix_c(matrix, var_name, filename):
    rows, cols = matrix.shape
    with open(filename, 'w') as f:
        f.write(f"#define {var_name}_ROWS {rows}\n")
        f.write(f"#define {var_name}_COLS {cols}\n")
        f.write(f"const float {var_name}[{var_name}_ROWS][{var_name}_COLS] = {{\n")
        for r in range(rows):
            row_vals = ", ".join(f"{v:.8f}f" for v in matrix[r])
            f.write(f"  {{{row_vals}}},\n")
        f.write("};\n")
    print(f"{filename} gerado: {rows}x{cols}")

export_matrix_c(mel_filterbank, "MEL_FILTERBANK", "mel_filterbank.h")
export_matrix_c(dct_matrix, "DCT_MATRIX", "dct_matrix.h")

## 20. Download de Todos os Artefatos Gerados

Esta seção final permite que você baixe todos os arquivos gerados durante o pipeline diretamente para sua máquina local. Esses arquivos são essenciais para a próxima etapa de desenvolvimento, que é a implantação no firmware do ESP32.

Os arquivos baixados incluem:

-   `modelo_sirene.onnx`: O modelo Keras exportado em formato ONNX.
-   `modelo_sirene_quant.tflite`: O modelo quantizado em formato TFLite Micro.
-   `modelo_sirene.h`: O arquivo de cabeçalho C contendo o modelo TFLite Micro como um array de bytes.
-   `normalizacao_params.h`: O arquivo de cabeçalho C com os parâmetros de normalização e quantização.
-   `mel_filterbank.h`: O arquivo de cabeçalho C com a matriz do banco de filtros mel.
-   `dct_matrix.h`: O arquivo de cabeçalho C com a matriz DCT para cálculo dos MFCCs.

Use `files.download()` da `google.colab` para baixar cada um desses arquivos. Eles serão cruciais para a construção do firmware embarcado.

In [ ]:
from google.colab import files

for filename in ["modelo_sirene.onnx", "modelo_sirene_quant.tflite",
                  "modelo_sirene.h", "normalizacao_params.h",
                  "mel_filterbank.h", "dct_matrix.h"]:
    files.download(filename)